In [32]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset

In [33]:
model = torch.load("../models/gnn_model.pt")

In [34]:
model

OrderedDict([('conv_layers.0.lin_key.weight',
              tensor([[-0.0113, -0.1622, -0.1866,  ...,  0.3240, -0.1502,  0.0351],
                      [ 0.0397,  0.1076,  0.0925,  ..., -0.0121,  0.0686,  0.1628],
                      [ 0.0239, -0.2063, -0.0098,  ...,  0.1056, -0.0382, -0.1045],
                      ...,
                      [-0.1598,  0.0031, -0.1803,  ..., -0.0122, -0.0129,  0.1275],
                      [-0.1540,  0.1999,  0.1297,  ..., -0.0216,  0.0184,  0.1129],
                      [-0.2051, -0.0506, -0.0150,  ..., -0.2196, -0.2111, -0.0628]])),
             ('conv_layers.0.lin_key.bias',
              tensor([ 0.1052, -0.1082, -0.1853, -0.1852, -0.0497, -0.1913,  0.1165,  0.1672,
                       0.1140, -0.1904,  0.1552,  0.1046, -0.0052, -0.1426, -0.0077, -0.0773,
                      -0.1819, -0.0983, -0.2083,  0.1137,  0.0486,  0.2050,  0.1333,  0.0585,
                       0.1378,  0.0104,  0.0922, -0.0219,  0.1932, -0.0275, -0.1365, -0.0221,


In [35]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [36]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads all graphs to a consistent node feature dimension."""

    def __init__(self, dataset, target_dim: int | None = None):
        self.dataset = dataset
        # Compute max dimension across entire dataset if not provided
        if target_dim is None:
            self.target_dim = self._compute_max_dim()
        else:
            self.target_dim = target_dim

    def _compute_max_dim(self) -> int:
        """Find the max node feature dimension across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            if hasattr(data, "x") and data.x.dim() > 1:
                max_dim = max(max_dim, data.x.shape[1])
        return max_dim

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        if hasattr(data, "x") and data.x.shape[1] < self.target_dim:
            pad_size = self.target_dim - data.x.shape[1]
            # Create a new copy to avoid mutating cache
            data = data.clone()
            data.x = torch.nn.functional.pad(data.x, (0, pad_size), mode="constant", value=0)
        return data

    def __len__(self) -> int:
        return len(self.dataset)

In [91]:
def build_train_val_test_loaders_two_stage(
    pt_paths: list[str],
    train_split: float = 0.8,
    val_within_train: float = 0.1,
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, DataLoader, DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    generator = torch.Generator().manual_seed(seed)
    primary_train_len = max(1, int(len(padded_dataset) * train_split))
    test_len = max(1, len(padded_dataset) - primary_train_len)
    while primary_train_len + test_len > len(padded_dataset):
        primary_train_len -= 1

    primary_train, test_ds = random_split(
        padded_dataset,
        [primary_train_len, test_len],
        generator=generator,
    )

    val_len = max(1, int(len(primary_train) * val_within_train))
    real_train_len = max(1, len(primary_train) - val_len)
    train_ds, val_ds = random_split(
        primary_train,
        [real_train_len, val_len],
        generator=generator,
    )

    pin_mem = torch.cuda.is_available()

    return (
        dataset,
        padded_dataset,
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [92]:
data_paths = collect_pt_paths("../outputs/data")

In [93]:
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 3000 .pt files for dataset.


In [94]:
for path in data_paths[:5]:  # Print first 5 paths as a sanity check
    print(path)

..\outputs\data\encoding_data_pennylane\random\random_Q10_L11_S1153511314.pt
..\outputs\data\encoding_data_pennylane\random\random_Q10_L11_S1239258084.pt
..\outputs\data\encoding_data_pennylane\random\random_Q10_L11_S1523000230.pt
..\outputs\data\encoding_data_pennylane\random\random_Q10_L11_S1617151728.pt
..\outputs\data\encoding_data_pennylane\random\random_Q10_L11_S1750066641.pt


In [117]:
dataset, padded_data, train_loader, val_loader, test_loader, node_in_dim, global_in_dim = build_train_val_test_loaders_two_stage(
    data_paths,  # Use first 10 paths for demonstration
    train_split=0.8,
    val_within_train=0.1,
    batch_size=32,
    seed=42,
    global_feature_variant="binned",
    node_feature_backend_variant=None,
)

In [118]:
padded_data

In [119]:
dataset

QuantumCircuitGraphDataset(3000)

In [120]:
for i, sample in enumerate(dataset):
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=167, edges=194, x_shape=(167, 23), y=5.797159671783447
   global features shape: torch.Size([1, 153])
1: nodes=171, edges=202, x_shape=(171, 23), y=5.6956353187561035
   global features shape: torch.Size([1, 153])
2: nodes=164, edges=188, x_shape=(164, 23), y=5.606733798980713
   global features shape: torch.Size([1, 153])
3: nodes=168, edges=196, x_shape=(168, 23), y=5.605652809143066
   global features shape: torch.Size([1, 153])
4: nodes=170, edges=200, x_shape=(170, 23), y=5.64307165145874
   global features shape: torch.Size([1, 153])
5: nodes=170, edges=200, x_shape=(170, 23), y=5.475927829742432
   global features shape: torch.Size([1, 153])
6: nodes=170, edges=200, x_shape=(170, 23), y=5.778342247009277
   global features shape: torch.Size([1, 153])
7: nodes=166, edges=192, x_shape=(166, 23), y=5.706170082092285
   global features shape: torch.Size([1, 153])
8: nodes=168, edges=196, x_shape=(168, 23), y=5.922117233276367
   global features shape: torch.Size([1, 153])
9

In [121]:
MASTER_GATE_TYPES = [
    "input", "measurement",
    "h", "s", "t", "id",
    "rx", "ry", "rz",
    "cx",
    "qx", "qy", "haar",
]

FAMILY_GATE_TYPES = {
    "random": ["input", "measurement", "rx", "ry", "rz", "cx"],
    "clifford": ["input", "measurement", "h", "s", "t", "id", "cx"],
    "haar": ["input", "measurement", "haar"],
    "quansistor": ["input", "measurement", "qx", "qy"],
}

In [122]:
class FamilyNodeProjector:
    def __init__(self, family: str):
        self.family = family
        self.master_gate_types = MASTER_GATE_TYPES
        self.family_gate_types = FAMILY_GATE_TYPES[family]
        self.keep_gate_idx = [
            self.master_gate_types.index(name) for name in self.family_gate_types
        ]
        self.n_gate_master = len(self.master_gate_types)

    def __call__(self, data: Data) -> Data:
        gate_part = data.x[:, :self.n_gate_master]
        qubit_part = data.x[:, self.n_gate_master:]

        out = data.clone()
        out.x = torch.cat([gate_part[:, self.keep_gate_idx], qubit_part], dim=1)
        return out

In [125]:
projector = FamilyNodeProjector("random")
for i, sample in enumerate(dataset):
    sample = projector(sample)
    x_shape = tuple(sample.x.shape) if getattr(sample, "x", None) is not None else None
    y_val = sample.y.item() if getattr(sample, "y", None) is not None and sample.y.numel() == 1 else getattr(sample, "y", None)
    print(f"{i}: nodes={sample.num_nodes}, edges={sample.num_edges}, x_shape={x_shape}, y={y_val}")
    print(f"   global features shape: {sample.global_features.shape if hasattr(sample, 'global_features') else None}")
    # print(f"   node feature sample: {sample.x[0] if hasattr(sample, 'x') and sample.x.shape[0] > 0 else None}")

    if i >= 9:
        print("...showing first 10 samples only")
        break

0: nodes=167, edges=194, x_shape=(167, 16), y=5.797159671783447
   global features shape: torch.Size([1, 153])
1: nodes=171, edges=202, x_shape=(171, 16), y=5.6956353187561035
   global features shape: torch.Size([1, 153])
2: nodes=164, edges=188, x_shape=(164, 16), y=5.606733798980713
   global features shape: torch.Size([1, 153])
3: nodes=168, edges=196, x_shape=(168, 16), y=5.605652809143066
   global features shape: torch.Size([1, 153])
4: nodes=170, edges=200, x_shape=(170, 16), y=5.64307165145874
   global features shape: torch.Size([1, 153])
5: nodes=170, edges=200, x_shape=(170, 16), y=5.475927829742432
   global features shape: torch.Size([1, 153])
6: nodes=170, edges=200, x_shape=(170, 16), y=5.778342247009277
   global features shape: torch.Size([1, 153])
7: nodes=166, edges=192, x_shape=(166, 16), y=5.706170082092285
   global features shape: torch.Size([1, 153])
8: nodes=168, edges=196, x_shape=(168, 16), y=5.922117233276367
   global features shape: torch.Size([1, 153])
9

In [115]:
sample

Data(
  x=[169, 16],
  edge_index=[2, 198],
  y=[1],
  global_features=[1, 153],
  num_qubits=10,
  gate_counts={
    CNOT_count=39,
    rx_bin_0=1,
    rx_bin_1=1,
    rx_bin_10=0,
    rx_bin_11=0,
    rx_bin_12=1,
    rx_bin_13=1,
    rx_bin_14=3,
    rx_bin_15=0,
    rx_bin_16=0,
    rx_bin_17=0,
    rx_bin_18=1,
    rx_bin_19=2,
    rx_bin_2=1,
    rx_bin_20=0,
    rx_bin_21=1,
    rx_bin_22=1,
    rx_bin_23=0,
    rx_bin_24=1,
    rx_bin_25=1,
    rx_bin_26=1,
    rx_bin_27=1,
    rx_bin_28=2,
    rx_bin_29=1,
    rx_bin_3=0,
    rx_bin_30=0,
    rx_bin_31=0,
    rx_bin_32=4,
    rx_bin_33=1,
    rx_bin_34=1,
    rx_bin_35=1,
    rx_bin_36=0,
    rx_bin_37=0,
    rx_bin_38=0,
    rx_bin_39=0,
    rx_bin_4=0,
    rx_bin_40=0,
    rx_bin_41=0,
    rx_bin_42=0,
    rx_bin_43=0,
    rx_bin_44=1,
    rx_bin_45=0,
    rx_bin_46=0,
    rx_bin_47=0,
    rx_bin_48=1,
    rx_bin_49=0,
    rx_bin_5=1,
    rx_bin_6=0,
    rx_bin_7=1,
    rx_bin_8=0,
    rx_bin_9=0,
    ry_bin_0=2,
    ry_bin_